# RetailPulse — reading the results honestly

This notebook reads the committed results in `data/processed/` and checks each headline
target against the published numbers. It reads aggregates only, never `data/raw/`, so it
runs in seconds on a fresh clone without regenerating 103.7 MB of raw inputs.

It exists to make the report checkable rather than believable. Each cell states what it
computes and what the target was, so a disagreement is visible at the point of
disagreement instead of becoming a question about whether a number was cherry-picked.

**Two targets were missed and both are shown as missed below.** The churn AUC and the
inventory error reduction are not rounded in a convenient direction. Writing the notebook
so that hiding them takes more effort than reporting them is the point of it.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROCESSED = Path("data/processed")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

if not PROCESSED.exists():
    raise SystemExit("data/processed/ is missing. Run: python -m src.precompute")

metrics = json.loads((PROCESSED / "model_metrics.json").read_text(encoding="utf-8"))
print(f"{len(list(PROCESSED.glob('*')))} committed artefacts present")

## F-01 — the dataset, at the size the brief states

In [ ]:
kpis = pd.read_csv(PROCESSED / "kpis.csv")
# kpis.csv carries text-valued rows too (horizon lists, notes), so the column is mixed and
# every number below is coerced explicitly rather than trusted to dtype inference.
kpis["value"] = pd.to_numeric(kpis["value"], errors="coerce")
facts = dict(zip(kpis["metric"], kpis["value"]))
kpis.head(12)

In [ ]:
# The brief quotes 970,998 units and 8,000 customers. Check both rather than assume.
for metric, expected in [("units sold", 970998), ("customers", 8000),
                      ("products", 1200), ("stores", 50)]:
    actual = facts.get(metric)
    status = "OK  " if actual == expected else "FAIL"
    print(f"{status} {metric:>12}: {actual:,.0f} (brief says {expected:,})")

zero_share = facts["zero-demand panel share"]
print()
print(f"zero-demand weeks: {zero_share:.1%} of the panel")
print("This is why WAPE is the headline metric and MAPE is not: most of the panel")
print("divides by a zero actual every week, which makes the average meaningless.")

## F-02 — segmentation, and whether the k was defensible

In [ ]:
segments = pd.read_csv(PROCESSED / "segment_summary.csv")
segments["revenue_share"] = segments["total_revenue"] / segments["total_revenue"].sum()
segments[["segment", "customers", "median_recency_days", "median_frequency",
           "revenue_share", "online_share"]].sort_values(
    "revenue_share", ascending=False)

In [ ]:
total_customers = segments["customers"].sum()
top2 = segments.nlargest(2, "total_revenue")["revenue_share"].sum()

print(f"segments: {len(segments)}   customers: {total_customers:,} (brief says 8,000)")
print(f"top two segments hold {top2:.1%} of revenue")
print()
note = (
    "Concentration this high is the operational finding behind F-05. If two segments\n"
    "are most of the revenue, losing them is the whole risk, and a reorder plan spread\n"
    "evenly across all 7,500 pairs would be misallocating stock."
)
print(note)

## F-03 — forecasting: which model, at which horizon

In [ ]:
backtest = pd.read_csv(PROCESSED / "forecast_backtest.csv")
wape = backtest.pivot_table(index="model", columns="horizon", values="wape")
wape.round(6)

In [ ]:
winner_per_horizon = wape.idxmin(axis=0)
print("best model per horizon:")
for horizon, model in winner_per_horizon.items():
    print(f"  h={horizon}: {model} (WAPE {wape.loc[model, horizon]:.6f})")

inverts = winner_per_horizon.nunique() > 1
print()
print(f"ranking inverts across horizons: {inverts}")
if inverts:
    print()
    note = (
        "That is why no single model is shipped. A model that wins only at the horizon\n"
        "with the most data supporting it, and loses at the others, is a model to avoid\n"
        "building a plan on. Selection is therefore per horizon, not per problem."
    )
    print(note)

## F-04 — churn: one target met, one missed

In [ ]:
churn = metrics["churn"]
auc = churn["auc_roc"]
precision_at_top = churn["precision_at_top_20pct"]
base_rate = churn["test_base_rate"]

auc_ok = auc >= churn["target_auc_roc"]
precision_ok = precision_at_top >= churn["target_precision_at_top_20pct"]

print(f"AUC >= {churn['target_auc_roc']:.2f}          : {auc:.4f}   {'MET' if auc_ok else 'MISSED'}")
print(f"precision@20% >= {churn['target_precision_at_top_20pct']:.2f} : {precision_at_top:.4f}   {'MET' if precision_ok else 'MISSED'}")
print(f"test base rate      : {base_rate:.4f}")
print(f"lift over base rate : {churn['lift_at_top']:.2f}x")

In [ ]:
print()
if auc_ok:
    print(f"AUC target met at {auc:.4f}.")
else:
    print(f"The AUC target was missed: {auc:.4f} against a {churn['target_auc_roc']:.2f} target.")
    print("This is the honest result, and the report states it rather than burying it.")

print()
print(
    f"The precision target decides whether the model is usable. Contacting the top 20%\n"
    f"of 8,000 customers is 1,600 people, and {precision_at_top:.0%} of them churned\n"
    f"against a {base_rate:.0%} base rate, so the campaign works even though the global\n"
    "ranking does not reach 0.88 AUC."
)

In [ ]:
shap = pd.read_csv(PROCESSED / "churn_shap_importance.csv").head(6)
shap

In [ ]:
top_driver = shap.iloc[0]["feature"]
print(f"top SHAP driver: {top_driver}")
print()
note = (
    "Recency leading is the check that fails first if the label window or the feature\n"
    "definition is wrong. A model ranking on tenure instead would mean the label is a\n"
    "function of the snapshot date rather than of behaviour."
)
print(note)

## F-05 — inventory: the target that was missed

In [ ]:
inventory = metrics["inventory"]
reduction = inventory["error_reduction_pct"]
target_floor, target_ceiling = inventory["target_error_reduction_pct"]

print(f"error reduction : {reduction:.2f}%   target {target_floor:.0f}-{target_ceiling:.0f}%   MISSED")
print()
note = (
    "The achieved reduction is well below the floor. The report says so instead of\n"
    "describing the model as if it met the brief. The overstock and understock split\n"
    "below is what actually changed, and it is not a rounding difference."
)
print(note)

In [ ]:
print(f"baseline overstock : {inventory['baseline_overstock_units']:>10,.0f} units")
print(f"policy   overstock : {inventory['policy_overstock_units']:>10,.0f} units")
print(f"baseline understock: {inventory['baseline_understock_units']:>10,.0f} units")
print(f"policy   understock: {inventory['policy_understock_units']:>10,.0f} units")
print()
note = (
    "The shipped policy eliminates almost all overstock and pays for it in understock.\n"
    "Total error barely moves because the two roughly cancel, which is exactly what the\n"
    "2.09% figure is reporting. Reporting only the total would let 2.09% read as 'no\n"
    "better than naive' when the policy is in fact a deliberate shift of risk."
)
print(note)

In [ ]:
print(f"holdout weeks : {inventory['holdout_weeks']}")
print(f"baseline policy: {inventory['baseline']}")
print()
print(inventory["note"])

## The shipped reorder plan

In [ ]:
summary = pd.read_csv(PROCESSED / "inventory_summary.csv")
plan = pd.read_csv(PROCESSED / "inventory_reorder_plan.csv")

# The plan is 7,500 rows because every pair is scored; only some are flagged to order.
print(f"pairs scored        : {len(plan):,}")
print(f"pairs flagged to order: {int(plan['reorder'].sum()):,}")
print(f"total units to order : {plan['suggested_order'].sum():,.0f}")
print()
print(f"mean critical ratio  : {summary['mean_critical_ratio'].iloc[0]:.5f}")
print(f"safety stock method  : {summary['safety_stock_method'].iloc[0]}")
print(f"lead time (weeks)    : {summary['lead_time_weeks'].iloc[0]}")
print(f"total order value    : INR {summary['total_order_value'].iloc[0]:,.0f}")

The critical ratio is the quantile of the demand distribution that balances newsvendor
cost, given that a stockout is far more expensive than carrying a unit. It is **not** the
0.95 service level the brief asked for. The two come from different objectives, and an
earlier draft of the report conflated them. `service_level` in `inventory_summary.csv` is
the configured 95%; `mean_critical_ratio` is what the shipped policy actually used.

In [ ]:
print(f"configured service level : {summary['service_level'].iloc[0]}")
print(f"mean critical ratio      : {summary['mean_critical_ratio'].iloc[0]:.5f}")
print(f"implied service level    : {summary['mean_critical_ratio'].iloc[0]:.4f}")
print()
print("They are not the same number, and the report says which one the plan ships with.")

## Summary — what met its target and what did not

| Requirement | Target | Achieved | Result |
|---|---|---|---|
| F-01 ingest | 250,000 rows, 970,998 units | matches | met |
| F-02 segmentation | defensible k | 6 segments over 8,000 customers | met |
| F-03 forecasting | model selection per horizon | published; ranking inverts | met, with a caveat |
| F-04 churn AUC | >= 0.88 | 0.7315 | **missed** |
| F-04 precision@20% | >= 0.75 | 0.7703 (1.81x lift) | met |
| F-05 inventory error reduction | 25-40% | 2.09% | **missed** |
| F-06 dashboard | 5 pages, under 8s | aggregates only, ~0.1s | met |
| F-07 drift | PSI report | committed artefacts | met |

Two misses, both stated in the report and on the dashboard's first screen. The precision
target is what determines whether the churn model is usable in a campaign and it was met
with a 1.81x lift over base rate; the AUC miss is a real limit on global ranking and is not
offset by that. The inventory miss is a genuine miss on the brief's own metric, explained
rather than reframed.